# Lab 1: The Agent Loop & Built-in Tools

## What is an Agent Loop?

An **agent loop** is a cycle where an LLM autonomously:
1. Receives a task
2. Reasons about what to do
3. Calls a tool (read a file, search, run a command)
4. Observes the result
5. Decides if it needs more information or can produce a final answer

The key insight: **you don't tell the agent which tools to call**. The model decides on its own based on the task description.

### Client SDK vs Agent SDK

| | Client SDK / Raw API | Agent SDK |
|---|---|---|
| **Who runs the loop?** | You do | The SDK does |
| **Tool execution** | You implement and dispatch | SDK handles automatically |
| **Use case** | Learning, custom workflows | Production agents |

In this lab, we build the loop ourselves to understand how it works under the hood.

---
## Setup

### Install Dependencies

| Package | Purpose |
|---------|---------|
| `openai` | OpenAI-compatible client (works with OpenRouter) |
| `python-dotenv` | Loads API keys from a `.env` file |

In [98]:
!pip install -q openai python-dotenv

### Import Libraries

We use `pathlib` for clean file operations, `re` for regex search, `openai` for API calls, and `dotenv` for key management.

In [99]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

### Load API Key from .env

Create a `.env` file in your project root with:
```
OPENROUTER_API_KEY=sk-or-v1-your-key-here
```

`load_dotenv()` reads this file and sets the environment variables. This keeps secrets out of the notebook.

In [100]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
print(f"API key loaded: {'Yes' if OPENROUTER_API_KEY else 'No'}")

API key loaded: Yes


---
## Step 1 — Define the Tools

Each tool has two parts:
1. **Function** — the actual Python code that runs
2. **JSON Schema** — describes the tool to the model (name, description, parameters)

The model reads the schema to understand what tools are available and how to call them.

### Tool 1: `read_file` — Read a file

Given a file path, return its contents. `pathlib.Path.read_text()` makes this a one-liner.

In [101]:
# pathlib makes this a one-liner
def read_file(path):
    """Read a file and return its contents."""
    return Path(path).read_text()

# The schema tells the model: "read_file takes one string argument called 'path'"
read_file_schema = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "Read the contents of a file.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path to the file"}
            },
            "required": ["path"]
        }
    }
}

### Tool 2: `glob` — Find files by pattern

Use glob patterns like `**/*.py` to find all Python files. `pathlib.Path.glob()` handles this natively.

In [102]:
# pathlib.Path.glob() handles recursion natively
def glob_files(pattern):
    """Find files matching a glob pattern."""
    return [str(p) for p in Path('.').glob(pattern)]

# The schema tells the model: "glob takes one string argument called 'pattern'"
glob_schema = {
    "type": "function",
    "function": {
        "name": "glob",
        "description": "Find files matching a glob pattern like **/*.py or data/**/*.py",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Glob pattern"}
            },
            "required": ["pattern"]
        }
    }
}

### Tool 3: `grep` — Search file contents

Search for regex patterns across files using `pathlib` + `re`. Returns matching lines with file paths and line numbers.

In [103]:
# pathlib + re + list comprehension keeps it compact
def grep_files(pattern, path):
    """Search file contents with regex. Returns list of (file, line_num, line) tuples."""
    regex = re.compile(pattern)
    results = []
    for f in Path('.').glob(path):
        if f.is_file():
            try:
                for i, line in enumerate(f.read_text().splitlines(), 1):
                    if regex.search(line):
                        results.append((str(f), i, line.strip()))
            except (UnicodeDecodeError, PermissionError):
                pass
    return results

# The schema tells the model: "grep takes two string arguments: pattern and path"
grep_schema = {
    "type": "function",
    "function": {
        "name": "grep",
        "description": "Search file contents with regex. Returns matching lines with file paths and line numbers.",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Regex pattern to search for"},
                "path": {"type": "string", "description": "Glob pattern for files to search (e.g. **/*.py)"}
            },
            "required": ["pattern", "path"]
        }
    }
}

### Register All Tools

Combine the schemas into a list (for the API) and a map (for dispatching calls).

In [104]:
# TOOLS: the list of schemas sent to the API
TOOLS = [read_file_schema, glob_schema, grep_schema]

# TOOL_MAP: maps tool names to functions for local execution
# When the model says "call read_file(path='data/app.py')", we look up read_file here
TOOL_MAP = {
    "read_file": read_file,
    "glob": glob_files,
    "grep": grep_files,
}

print(f"Tools registered: {list(TOOL_MAP.keys())}")

Tools registered: ['read_file', 'glob', 'grep']


---
## Step 2 — Configure the Client

We use OpenRouter's OpenAI-compatible API. The client talks to `https://openrouter.ai/api/v1` using your API key.

We also define the model name and target directory for the scan.

In [105]:
# OpenRouter exposes an OpenAI-compatible endpoint
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Free model from NVIDIA via OpenRouter
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

# The fixture codebase shipped with this lab
TARGET_DIR = "data"

print(f"Model: {MODEL}")
print(f"Target: {TARGET_DIR}")

Model: nvidia/nemotron-3-ultra-550b-a55b:free
Target: data


---
## Step 3 — Define the Task

The task is a natural-language prompt. It tells the agent **what** to do, not **how**.

The model will decide which tools to call based on this prompt.

In [106]:
TASK = f"""
Scan the codebase at {TARGET_DIR} and find all TODO and FIXME comments.

For each match, report:
- File path
- Line number
- The comment text
- A brief note on what the comment is about

Organize the results as a markdown summary grouped by file.
"""

---
## Step 4 — Run the Agent Loop

This is the core of the lab. Here's the loop step by step:

```
1. Send messages + tool definitions to the model
2. Model responds with text OR a tool call
3. If tool call → execute function locally → append result → go to 2
4. If text → done, return final answer
```

The `messages` list grows with each iteration. The model sees the full history:
- What it said before
- What tools it called
- What results it got

This is exactly what the Agent SDK does for you automatically with `query()`.

In [107]:
# Initialize the conversation with a system prompt and the user task
messages = [
    {"role": "system", "content": "You are a code exploration assistant. Scan codebases, find patterns, and produce structured markdown reports. Be thorough but concise. Always cite file paths and line numbers."},
    {"role": "user", "content": TASK},
]

tool_call_count = 0
MAX_ITERATIONS = 15  # Safety limit to prevent infinite loops

# --- THE AGENT LOOP ---
for i in range(MAX_ITERATIONS):

    # 1. Send messages to the model (with tool definitions)
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
    )

    choice = response.choices[0]
    message = choice.message

    # 2. Check if the model wants to call a tool
    if message.tool_calls:
        # The model decided to use a tool — append its message to history
        messages.append(message)

        # 3. Execute each tool call locally
        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            tool_call_count += 1

            print(f"  [Call {tool_call_count}] {func_name}({func_args})")

            # Look up the function and call it with the model's arguments
            if func_name in TOOL_MAP:
                result = TOOL_MAP[func_name](**func_args)
            else:
                result = f"Unknown tool: {func_name}"

            # Convert result to string (the API expects string content)
            result_str = json.dumps(result, default=str)

            # Append the tool result so the model can see it in the next iteration
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result_str,
            })

    else:
        # 4. No tool calls — the model produced a final text answer
        final_answer = message.content
        break

print(f"\nDone. Tool calls made: {tool_call_count}")

  [Call 1] grep({'path': 'data/**/*', 'pattern': 'TODO|FIXME'})
  [Call 2] read_file({'path': 'data/tests/test_helpers.py'})
  [Call 3] read_file({'path': 'data/src/app.py'})
  [Call 4] read_file({'path': 'data/src/core/services/auth.py'})
  [Call 5] read_file({'path': 'data/src/utils/helpers.py'})
  [Call 6] read_file({'path': 'data/src/api/client.py'})

Done. Tool calls made: 6


### Display the Agent Response

In [108]:
print(f"{'='*50}")
print("AGENT RESPONSE:")
print(f"{'='*50}\n")
print(final_answer)

AGENT RESPONSE:

# TODO/FIXME Comments Report

## Summary
- **Total files scanned:** 5
- **Total TODO comments:** 8
- **Total FIXME comments:** 5
- **Note:** Comments inside string literals are excluded from this report.

---

## `data/tests/test_helpers.py`

| Line | Type | Comment | Context |
|------|------|---------|---------|
| 7 | TODO | Add more edge cases for special characters | In `test_sanitize_input()` - suggests expanding test coverage for the sanitize_input function |
| 13 | FIXME | Fails for .museum TLD | In `test_validate_email()` - indicates the email validation regex doesn't handle longer TLDs like `.museum` |

> Lines 18-19 contain TODO/FIXME references in strings/comments but are not actionable code comments.

---

## `data/src/app.py`

| Line | Type | Comment | Context |
|------|------|---------|---------|
| 5 | TODO | Add input validation before processing | In `process_request()` - validation needed before calling `transform()` |
| 11 | FIXME | This breaks when da

### What just happened?

The agent loop ran multiple iterations:
1. The model received the task and decided to search for files using `glob`
2. It then `grep`ped for TODO/FIXME patterns across matching files
3. It may have used `read_file` to get context around specific matches
4. After gathering enough data, it produced the final markdown summary

You never told the agent which files to read or which tools to call — it figured that out autonomously.

---
## Step 5 — Result Summary

Parse the agent's free-form text output to extract structured metrics.
This shows how you can post-process LLM output with regex.

In [109]:
# Count occurrences of each keyword in the agent's response
todo_count = len(re.findall(r'(?i)TODO', final_answer))
fixme_count = len(re.findall(r'(?i)FIXME', final_answer))

# Count unique .py files mentioned
file_mentions = len(set(re.findall(r'\b[\w/]+\.py\b', final_answer)))

print("="*50)
print("RESULT SUMMARY")
print("="*50)
print(f"  TODO mentions:    {todo_count}")
print(f"  FIXME mentions:   {fixme_count}")
print(f"  Unique files:     {file_mentions}")
print("="*50)

RESULT SUMMARY
  TODO mentions:    13
  FIXME mentions:   11
  Unique files:     5


---
## Step 6 — LLM Judge

Use a **second LLM call** to evaluate the agent's output.
This is a common pattern: one LLM generates, another evaluates.

The judge checks:
- **Coverage** — Did it find all TODO/FIXME comments?
- **Accuracy** — Are reported items real comments (not false positives from strings)?
- **Completeness** — Did it include file paths and line numbers?
- **Format** — Is the output well-organized?

In [110]:
# Build the judge prompt with the agent's output embedded
judge_prompt = f"""
You are an evaluation judge. Analyze the following agent output and the codebase.

AGENT OUTPUT:
{final_answer}

TASK: Find all TODO and FIXME comments in the codebase.

Evaluate on these criteria:
1. COVERAGE: Did the agent find all the TODO/FIXME comments?
2. ACCURACY: Are all reported items real TODO/FIXME comments (not false positives from strings)?
3. COMPLETENESS: Did it include file paths and line numbers?
4. FORMAT: Is the output well-organized and readable?

Score each criterion 1-5 and give an overall score. Be strict.
"""

# Single API call — no tools, just text generation
judge_response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": judge_prompt}],
)

judge_content = judge_response.choices[0].message.content

print("="*50)
print("LLM JUDGE EVALUATION")
print("="*50)
print(judge_content if judge_content else "(No response from judge — model may have returned empty content)")

LLM JUDGE EVALUATION
{
  "coverage": 5,
  "accuracy": 5,
  "completeness": 5,
  "format": 5,
  "overall": 5,
  "analysis": "The agent output appears to be a comprehensive and accurate report of TODO/FIXME comments across the codebase. It correctly identifies 5 files scanned, with specific line numbers, types (TODO vs FIXME), comment text, and context for each. The agent properly excludes comments inside string literals and dictionary values (noted in the report for files where this occurs). The priority recommendations table is well-organized with appropriate severity levels. The format is clean, readable, and structured with summary statistics, per-file breakdowns, and actionable recommendations."
}


---
## Try It Yourself

Change `TARGET_DIR` and `TASK` above and re-run from **Step 4**.

Try different patterns like `HACK`, `XXX`, `DEPRECATED`, or `@deprecated`.